<a href="https://colab.research.google.com/github/solasobambo-prog/ts-data-science/blob/main/Week3a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    'customer_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, np.nan],
    'age':     [34, np.nan, 29, 41, np.nan, 52, 38, np.nan, 27, 45, 31, 36],
    'city':    ['Lagos', 'Abuja', None, 'Kano', 'Lagos', None,
                'Ibadan', 'Abuja', 'Lagos', 'Kano', None, 'Lagos'],
    'income':  [250000, np.nan, 180000, np.nan, np.nan, 420000,
                np.nan, 150000, np.nan, 310000, np.nan, 205000],
    'revenue': [1200, 450, 800, 2300, 300, 5100, 640, 275, 980, 3050, 410, 760],
    'churned': ['No', 'Yes', 'No', 'No', 'Yes', 'No',
                'Yes', 'No', 'No', 'No', 'Yes', 'No'],
})

In [3]:
# 1. Count the missing values per column
print("--- MISSING VALUE COUNTS ---")
print(customers.isna().sum())

# 2. Show the missing percentage per column (rounded to 1 decimal place)
print("\n--- MISSING VALUE PERCENTAGES (%) ---")
print((customers.isna().mean() * 100).round(1))

# 3. Show rows where age is missing
print("\n--- ROWS WITH MISSING AGE ---")
display(customers[customers['age'].isna()])


--- MISSING VALUE COUNTS ---
customer_id    1
age            3
city           3
income         6
revenue        0
churned        0
dtype: int64

--- MISSING VALUE PERCENTAGES (%) ---
customer_id     8.3
age            25.0
city           25.0
income         50.0
revenue         0.0
churned         0.0
dtype: float64

--- ROWS WITH MISSING AGE ---


,customer_id,age,city,income,revenue,churned
1,102.0,NaN,Abuja,NaN,450,Yes
4,105.0,NaN,Lagos,NaN,300,Yes
7,108.0,NaN,Abuja,150000.0,275,No


In [4]:
# --- DROP ---
# Prediction: The initial row count is 12. Since only 1 row has a missing customer_id, the new count will be 11.
# Safe because: 'customer_id' is the primary identifier for a customer; a record without it cannot be accurately linked or tracked.
customers = customers.dropna(subset=['customer_id']).copy()


# --- FILL ---
# Expected Age Median: 34.0 (The non-missing ages are. Sorted: 27, 29, 31, 34, 36, 38, 41, 45, 52. Median is 36.
# Justification (Age): Median is robust against potential extreme age outliers and preserves the central tendency without shifting the distribution.
# Justification (City): 'Unknown' prevents losing data while explicitly categorizing categorical rows where the location is missing.
customers['age'] = customers['age'].fillna(customers['age'].median())
customers['city'] = customers['city'].fillna('Unknown')


# --- FLAG ---
# Prediction: Missing income likely correlates with higher churn rates, as non-reporting or irregular income data can signal unengaged or high-risk profiles.
customers['income_missing'] = customers['income'].isna()

print("\n--- CHURN RATE COMPARISON BY INCOME MISSING STATUS ---")
churn_comparison = (customers['churned'] == 'Yes').groupby(customers['income_missing']).mean() * 100
display(churn_comparison)

# Decision on Income: Income should be kept with the flag column rather than dropped.
# Reason: Dropping income entirely discards a vital numeric feature for 5 out of 11 rows (~45% of the data), while dropping rows would shrink our sample size drastically. Keeping the flag captures the behavioral pattern that missingness itself correlates with churn.



--- CHURN RATE COMPARISON BY INCOME MISSING STATUS ---


,churned
income_missing,
False,0.000000
True,66.666667


In [5]:
# Build a comprehensive missing value report sorted from most missing to least
missing_report = pd.DataFrame({
    'missing_count': customers.isna().sum(),
    'missing_pct': (customers.isna().mean() * 100).round(1)
}).sort_values(by='missing_count', ascending=False)

print("--- FINAL MISSING VALUE REPORT ---")
display(missing_report)


--- FINAL MISSING VALUE REPORT ---


,missing_count,missing_pct
income,6,54.5
age,0,0.0
customer_id,0,0.0
city,0,0.0
revenue,0,0.0
churned,0,0.0
income_missing,0,0.0
